
# HTTP Streaming and Retry Strategies

A practical demonstration of incremental HTTP responses,
bounded retries, exponential backoff, and quota-aware error handling.

**Technologies:** Python, `http.server`, `requests`, and Tenacity.

### Course references

Adapted from the following notebooks in the
[BrightPool Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course):

- [Streaming with Responses API](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/streaming_with_responses_api.ipynb)
- [Rate Limits and Retrying with Tenacity](https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/rate_limits_and_retrying_with_tenacity.ipynb)

### Adaptations

The course demonstrates OpenAI streaming and API retry behaviour.

This notebook uses a local HTTP server to demonstrate actual
incremental data transfer, and simulated exceptions to test
retry policies without sending model requests.

**No Gemini or OpenAI API key is required.**

The streaming example demonstrates the underlying HTTP concept,
not actual LLM token streaming or OpenAI Responses API events.



## 1. Setup

Install the libraries required for HTTP requests and retry handling.

- `requests` sends HTTP requests.
- `tenacity` provides retry decorators and backoff policies.
- `http.server` creates a local HTTP server.
- `threading` allows the server to run while the client receives data.

All streaming traffic stays within the Colab runtime.
No external AI service is contacted.


In [1]:

%pip install -q requests tenacity

import time
import requests

from http.server import BaseHTTPRequestHandler, HTTPServer
from threading import Thread

from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential
)

print("Setup completed.")


Setup completed.



## 2. Understanding Streaming

Without streaming, a client generally receives the response
after the server has completed generating it.

With streaming, the client can process incoming data
incrementally while the connection remains open.

This is useful for:
- AI chat applications displaying generated text progressively
- Large downloads
- Real-time server updates

Our demonstration uses a local server that sends four
text fragments with short delays.

HTTP transport does not guarantee that the client receives
exactly the same chunk boundaries as those used by the server.


In [2]:

# Predefined chunks: no HTTP connection yet.

chunks = [
    "Peter Piper ",
    "picked a peck ",
    "of pickled peppers."
]

received = []

for chunk in chunks:
    print(chunk, end="", flush=True)
    received.append(chunk)

print("\nReconstructed:", "".join(received))


Peter Piper picked a peck of pickled peppers.
Reconstructed: Peter Piper picked a peck of pickled peppers.



## 3. Live HTTP Streaming

The next example creates a local HTTP server and connects
to it using `requests`.

The server sends text in several writes with short delays.
The client reads and prints the incoming bytes incrementally.

Unlike the previous example, this uses an actual HTTP connection.

LLM APIs may use a related streaming mechanism such as
Server-Sent Events (SSE), with provider-specific event formats.

**This cell does not call an LLM or consume API quota.**


In [3]:

# Real HTTP streaming — no Gemini API or API key required

class StreamHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200)
        self.end_headers()

        for text in ["Hello! ", "This is ", "a live ", "HTTP stream."]:
            self.wfile.write(text.encode())
            self.wfile.flush()
            time.sleep(0.5)

    def log_message(self, *args):
        pass


server = HTTPServer(("127.0.0.1", 0), StreamHandler)
Thread(target=server.serve_forever, daemon=True).start()

try:
    with requests.get(
        f"http://127.0.0.1:{server.server_port}",
        stream=True,
        timeout=10
    ) as response:
        response.raise_for_status()

        for chunk in response.iter_content(chunk_size=1):
            print(chunk.decode(), end="", flush=True)

finally:
    server.shutdown()
    server.server_close()


Hello! This is a live HTTP stream.


## 4. Rate Limits and Retry Strategies

API requests can fail for different reasons.

Common HTTP status codes include:

- **429 Too Many Requests:** The client has exceeded a rate
  limit or quota.
- **503 Service Unavailable:** The server is temporarily
  unable to process the request.
- **500 Internal Server Error:** An unexpected server-side
  failure occurred.

A retry means attempting the operation again after failure.

Retries should be:
1. Limited to appropriate error types
2. Bounded by a maximum number of attempts
3. Delayed using a suitable backoff policy

Daily quota exhaustion should not trigger repeated retries.


In [5]:

# Simulated errors — no real API requests

class TemporaryServerError(Exception):
    """Simulates a temporary server failure."""


class DailyQuotaExceeded(Exception):
    """Simulates exhausted daily API quota."""


attempts = {"count": 0}


@retry(
    retry=retry_if_exception_type(TemporaryServerError),
    stop=stop_after_attempt(3),
    wait=wait_exponential(
        multiplier=0.1,
        min=0.1,
        max=0.5
    ),
    reraise=True
)
def flaky_operation():
    attempts["count"] += 1

    print("Attempt:", attempts["count"])

    if attempts["count"] < 3:
        raise TemporaryServerError("Simulated HTTP 503")

    return "Operation succeeded"


result = flaky_operation()

print("Result:", result)
print("Total attempts:", attempts["count"])


Attempt: 1
Attempt: 2
Attempt: 3
Result: Operation succeeded
Total attempts: 3



## 5. Do Not Retry Exhausted Daily Quotas

Not every failure should trigger a retry.

A temporary HTTP 503 may justify a limited retry.

However, when an API reports that its daily request
quota is exhausted, repeatedly sending the same
request will not restore quota.

This example confirms that the retry policy stops
immediately for a different exception type.


In [6]:

quota_attempts = {"count": 0}


@retry(
    retry=retry_if_exception_type(TemporaryServerError),
    stop=stop_after_attempt(3),
    wait=wait_exponential(
        multiplier=0.1,
        min=0.1,
        max=0.5
    ),
    reraise=True
)
def quota_exhausted_operation():
    quota_attempts["count"] += 1

    print("Attempt:", quota_attempts["count"])

    raise DailyQuotaExceeded(
        "Daily API quota exhausted"
    )


try:
    quota_exhausted_operation()

except DailyQuotaExceeded as error:
    print("Error:", error)


print("Total attempts:", quota_attempts["count"])


Attempt: 1
Error: Daily API quota exhausted
Total attempts: 1



## 6. Production Considerations

A production application must distinguish between
temporary failures and permanent or quota-related failures.

Important considerations include:

- Classify errors using actual HTTP responses and
  provider-specific error details.
- Respect server-provided retry guidance where available.
- Use bounded exponential backoff, potentially with jitter,
  for genuinely transient failures.
- Avoid automatically retrying exhausted daily quotas.
- Consider whether repeating a request is safe. Operations
  that modify data may require idempotency safeguards.
- Log failed attempts without exposing API keys or secrets.

A timeout does not necessarily mean that the server
did not process the request. Retrying can therefore
cause additional usage or duplicate actions.

The examples in this notebook demonstrate retry-policy
behaviour, not a complete production error classifier.



## 7. Findings

### Streaming

The first example processed predefined text chunks.
The second example demonstrated actual incremental
data transfer over a local HTTP connection.

Streaming allows applications to display data before
the entire response has been received.

### Retry handling

The transient-error simulation succeeded on the third
attempt using a bounded retry policy.

The daily-quota simulation stopped after one attempt,
showing that retry behaviour can be restricted by
exception type.

### Limitations

- No live LLM token streaming was performed.
- The streaming server and client ran locally.
- The retry examples used simulated exceptions.
- No actual API response headers or quota information
  were inspected.
- Real systems need provider-specific error classification,
  appropriate retry delays, and additional testing.

The notebook demonstrates HTTP streaming and reliability
principles without requiring a Gemini or OpenAI API key.


In [7]:

assert attempts["count"] == 3
assert quota_attempts["count"] == 1

print("Retry policy checks passed.")
print("Gemini API requests used: 0")


Retry policy checks passed.
Gemini API requests used: 0
